# DICE — Notebook 1.2 (Scenarios & Forcing) — Student version


**Learning objectives (1h30min)**
0. Chargez un code DICE minimal et identifiez ses **blocs**: paramètres, états initiaux, équations économiques, équations climatiques.
1. Scénarios SSP : générer des scénarios SSP significatifs en ajustant l'étalonnage des variables exogènes.
2. Introduce some forcing objectives, by adjusting

> Ce carnet utilise le`DICE.py` module (teaching version).


# 0) Charger DICE et exécuter le niveau de référence

In [ ]:
# Run this cell once to check/install the Python packages required for this notebook.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
# 1) Load DICE and baseline run
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
) 

p = Params()
path = init_states(p)
path[:, p.i_mu] = 0.03          # baseline abatement path
path = update_path(path, range(1, p.nT), p)
df_base = mat_to_df(path, p)
df_base.head()


## 1) SSP Scenarios

Dans ce qui suit, nous voulons simuler deux narrations extrêmes, les narrations **SSP1** et **SSP5**, ainsi que la **référence (SSP2)**.

- **SSP1** – * Durabilité : prendre la route verte (faibles défis à l'atténuation et à l'adaptation)*
  *Le monde évolue progressivement vers une voie plus durable, mettant l'accent sur le développement inclusif et le respect des frontières environnementales. Les investissements dans l'éducation et la santé accélèrent la transition démographique, l'inégalité est réduite et les modes de consommation évoluent vers une faible croissance matérielle et une moindre intensité des ressources et de l'énergie*.

- **SSP5** – *Le développement à combustible fossile : prendre la route (Principaux défis à l'atténuation, faibles défis à l'adaptation)*
  *Ce monde met l'accent sur les marchés concurrentiels, l'innovation et le développement du capital humain. La croissance économique rapide est alimentée par les ressources fossiles et les modes de vie à forte intensité énergétique. Les problèmes environnementaux locaux sont gérés, mais les biens communs mondiaux sont soulignés, avec des émissions élevées et une dépendance à l'égard de futures solutions technologiques ou géo-ingénierie*.

Nous examinons donc trois scénarios : le **optimiste** (SSP1), le **référence** (SSP2, calibrage du courant) et le **pessimiste** (SSP5).
Le tableau ci-dessous résume les modifications proposées pour l'étalonnage de base :

Paramètre (description) Symbole SSP1 (Optimiste) SSP2 (Base – DICE) SSP5 (Pessimiste)
|----------------------------------------|----------|--------------------------------------------|-------------------------|-------------------------------------------|
Intensité des émissions (initiale)$\sigma_0$ Valeur étalonnée de 25% inférieure à la valeur de référence
Diminution de l ' intensité des émissions$g_{\sigma}$ | 25% faster decline                         | calibrated value        | 50% slower decline                        |
| Long-run population (asymptote)        | $L_{\infty}$ | 9 billion                                  | calibrated value        | 16 billion                                |
paramètre de croissance démographique$l_g$      | 20% lower                           | calibrated value        | 30% higher                        |
| Total factor productivity (TFP) level  | $A_0$      Valeur étalonnée de 5 % supérieure à la valeur de référence
Taux de croissance TFP$g_A$      15% de plus que le niveau de référence

> **Instructions**: Pour chaque scénario, initialisez les paramètres en conséquence, exécutez le modèle et comparez les trajectoires pour la production, les émissions, le forçage et les températures.


#### 1-A) Construisez l'étalonnage et vérifiez qu'il fonctionne dans un nouveau vecteur p.
Utilisez le tableau pour remplir chaque scénario et avoir la bonne narration.

In [ ]:
# Build parameter variants from the calibration table above.
def make_params_variant(**kwargs):
    q = Params()
    for name, value in kwargs.items():
        setattr(q, name, value)
    return q

p_base = Params()

# Each narrative changes all six drivers below, not only productivity:
# sigma0, gsig, lg, Linf, A0 and gA.
# Replace the ellipses using the optimistic and pessimistic table multipliers.
p_opt = make_params_variant(
    sigma0=..., gsig=..., lg=..., Linf=..., A0=..., gA=...
)
p_pes = make_params_variant(
    sigma0=..., gsig=..., lg=..., Linf=..., A0=..., gA=...
)


#### 1-B) Fournir un graphique comparant les émissions, le PIB, les dommages (les définir manuellement) et les températures.

In [ ]:
# Your code for question 1-B should be here

### 1-B) Report the graph and compare them
sim_base = init_states(p_base)
#sim_opt = init_states(p_opt)
#sim_pes = init_states(p_pes)


sim_base[1:, p_base.i_s]  = 0.20  # constant saving rate at 20%
sim_base[1:, p_base.i_mu] = 0.03
timevec = range(1, p_base.nT)
sim_base = update_path(sim_base, timevec, p_base)


#### 1-C) Selon tous les chiffres, pensez-vous que le récit des SSP sont les principaux moteurs du changement climatique? Quelles sont les températures dans le meilleur cas du changement climatique par rapport au pire?

> ✍️ You written answer here.

## 2) Calendriers d'atténuation: réduction de la rampe pour atteindre (près) zéro net d'ici 2100 / 2080 / 2060

Jusqu'à présent, nos scénarios étaient purement **exogènes** : nous avons varié la productivité, la population et l'intensité des émissions pour imiter différents récits SSP.
Nous enrichissons maintenant l'analyse en introduisant une variable **deuxième décision**: le taux de **abattement**$\mu_t$, qui mesure la fraction des émissions industrielles évitées par les politiques et les technologies d'atténuation.

- Dans les exercices **référence**,$\mu_t$ was fixed at a constant, low level (e.g. 3%).  
- Dans cet exercice, nous alimentons explicitement une trajectoire de **abatement**$\{\mu_t\}$ dans le modèle.
- Lorsque $\mu_t \to 1$, presque toutes les émissions industrielles de CO₂ sont éliminées. Les émissions liées aux sols $E_{\text{land},t}$ restent toutefois exogènes et peuvent demeurer positives.

Cela reflète la façon dont les scénarios sont construits en **CMIP6 (phase 6 du projet d'intercomparaison de modèles couplés)**, où chaque scénario SSP peut être combiné à des hypothèses politiques différentes **climates**:
- Une voie d'atténuation ** à haut niveau d'ambition** permet de réduire rapidement les émissions, atteignant ** zéro émission** au milieu du siècle (p. ex. 2060).
- A **delayed mitigation pathway** reaches net-zero much later (e.g. 2100).  
- Les niveaux **forçants par 2100** (par exemple 2,6, 4,5, 8,5 W/m2) définissent les scénarios bien connus **SSP1-2.6**, **SSP2-4.5** et **SSP5-8.5**.


3. Interpréter les résultats à la lumière du cadre de scénarios SSP du GIEC : *décarbonisation précoce et ambitieuse (SSP1-2.6) vs action retardée avec un réchauffement élevé (SSP5-8.5)*.

#### 2-a) Construire des rampes linéaires pour$\mu_t$ qui atteignent **net zéro** en 2100, 2080 et 2060.
Rappelons que dans ce modèle, nous avons seulement deux variables de décision ici. Les économies restent inchangées$\{s_t\}_{t=t_{0}}^{T}=0.2$ while mitigation path must be adjusted: $\{\mu_t\}_{t=t_{0}}^{T}$. Il devrait augmenter progressivement à 1 pour atteindre zéro net à une date donnée.
Recall that here, our simulations, including decision variables, are all stacked into matrix $w = [y,x,z]$. In what follows, we modify directly $w$.



In [ ]:
# Construct date-based ramps so the code remains valid if Delta changes.
def abatement_ramp(years, target_year):
    ramp = np.zeros(len(years))
    target = int(np.argmin(np.abs(years - target_year)))
    # Fill indices 1...target with np.linspace(0.0, 1.0, target),
    # then keep all subsequent values equal to 1.0.
    return ramp

years = sim_base[:, p_base.i_time]
ramp2060 = abatement_ramp(years, 2060)
ramp2080 = abatement_ramp(years, 2080)
ramp2100 = abatement_ramp(years, 2100)

# Plot the three ramps before using them in the model.


#### 2-b) Simuler le modèle sous chaque rampe et comparer les chemins pour **forçage$F_t$**, **temperatures $T_{AT,t}$**, et **émissions$E_t$**.
Rappelons qu'une fois la matrice de décision$z$ de taille$I\times 2$ a été modifié (dans le cadre$w$), nous pouvons à nouveau résoudre le nouveau système avec une politique climatique alternative:
  $$
  y_t = f_p (y_{t-1}, x_{t-1}, z_t), \quad f: \mathbb{R}^{N_y}\times \mathbb{R}^{N_x}\times \mathbb{R}^2 \to \mathbb{R}^{N_y}.
  $$

  Quelle est l'efficacité de la politique climatique dans la réduction des émissions?

In [ ]:
# Use the same three steps for every policy path:
# 1. copy sim_base;
# 2. assign the relevant ramp to path[:, p_base.i_mu];
# 3. call update_path(path, timevec, p_base).

sim2060 = sim_base.copy()
sim2060[:, p_base.i_mu] = ramp2060
# sim2060 = ...

# Repeat for sim2080 and sim2100, then compare i_F, i_T_AT and i_E.


> ✍️ You written answer here.

### 2-c) Quel est le coût économique de l'atténuation du climat?
Fournir 2 parcelles, l'une montrant l'écart de consommation défini comme$100\times (c_t^{alt}/c_t^{base}-1)$ et dans une deuxième figure l'écart de température$(T_t^{alt}-T_t^{base})$. $T_t^{alt}$ indique la trajectoire de température dans d'autres régimes (p. ex. optimiste ou pessimiste) alors que *base* est la base de référence.
Commentez le résultat de votre recherche sur le sacrifice qui doit être fait par la société en termes de perte de consommation pendant la transition. Pourquoi ce sacrifice ?

In [ ]:
# Example for net zero in 2060:
# gapC_2060 = 100 * (sim2060[:, p_base.i_C] / sim_base[:, p_base.i_C] - 1)
# gapT_2060 = sim2060[:, p_base.i_T_AT] - sim_base[:, p_base.i_T_AT]

# Repeat for 2080 and 2100. Display consumption and temperature gaps
# in two separate panels.


> ✍️ You written answer here.

## 3) Providing core scenarios mixing SSPs $\times$ Forcing as in IPCC


Le tableau ci-dessous montre comment **SSP storylines** se combine avec **objectifs de forçage radiatif** (W/m2 par 2100).
Par exemple, *SSP1-2.6* correspond au narratif **SSP1** combiné à une voie de forçage **2.6 W/m2**.

|      | **1.9** | **2.6** | **4.5** | **7.0** | **8.5** |
|------|---------|---------|---------|---------|---------|
| **SSP1** | SSP1-1.9 | SSP1-2.6 | –       | –       | –       |
| **SSP2** | –       | SSP2-2.6 | SSP2-4.5 | SSP2-7.0 | –       |
| **SSP3** | –       | –       | SSP3-4.5 | SSP3-7.0 | SSP3-8.5 |
| **SSP4** | –       | SSP4-2.6 | SSP4-4.5 | SSP4-7.0 | –       |
| **SSP5** | –       | –       | SSP5-4.5 | –       | SSP5-8.5 |

### Approximate Mapping: Forcing → Global Warming by 2100

Les voies de forçage radiatif se traduisent par des augmentations de température moyennes** différentes par rapport aux niveaux préindustriels (1850–1900).
Les valeurs ci-dessous sont **estimations centrales approximatives** (IPCC AR6, hypothèses médianes de sensibilité au climat):

- **1,9 W/m2 (SSP1-1.9)** → ~ **1,5 °C** stabilisation (conforme à l'objectif de l'Accord de Paris de 1,5 °C).
- **2.6 W/m² (SSP1/2/4-2.6)** → ~ **2.0 °C** warming by 2100 (low-forcing pathway).  
- **4.5 W/m² (SSP2/3/4/5-4.5)** → ~ **2.5–3.0 °C** warming by 2100 (intermediate pathway).  
- **7.0 W/m² (SSP2/3/4-7.0)** → ~ **3.5–4.0 °C** warming by 2100 (high-forcing pathway).  
- **8.5 W/m2 (SSP3/5-8.5)** → ~ **4.5 °C ou plus** réchauffement de 2100 (émissions très élevées, référence du pire cas).

---

👉 **Interpretation**:  
- Les voies de forçage plus basses (SSP1-1.9, SSP1/2-2.6) nécessitent ** une atténuation précoce et ambitieuse** et correspondent généralement au CO2 net zéro au milieu du siècle.
- Les voies de forçage plus élevées (SSP3-7.0, SSP5-8.5) supposent ** une atténuation limitée**, une utilisation continue des combustibles fossiles et des transitions nettes-zéro retardées ou absentes.
- Les cas intermédiaires (SSP2-4.5) représentent **les trajectoires actuelles des politiques** ou les efforts d'atténuation retardés.

#### 3-A) Certains scénarios de base en dehors de la diagonale ne sont pas évalués (p. ex. SSP1-8.5). Pourquoi ?

> ✍️ You written answer here.

#### 3-B) Essayez de calculer le SSP1-1.9, qui correspond au zéro net d'ici 2060 et au scénario optimiste. Le SSP1-1.9 est-il hors de portée?

In [ ]:
# SSP1-like assumptions with net zero by 2060:
# 1. sim2060_opt = init_states(p_opt)
# 2. reuse the baseline saving path;
# 3. assign ramp2060 to p_opt.i_mu;
# 4. call update_path once with p_opt;
# 5. compare forcing, temperature and emissions and report 2100 warming.


> ✍️ You written answer here.

#### 3-C) Est-il possible d'obtenir 4°C de réchauffement dans le SSP1? Évaluer SSP1 en l'absence de politique d'atténuation$\mu=0$ pour imiter SSP1-8.5

In [ ]:
# SSP1-like / no-mitigation stress test:
# start from init_states(p_opt), reuse the baseline saving path,
# set abatement to zero, then call update_path with p_opt.
# Compare with sim_base and report atmospheric warming in 2100.


> ✍️ You written answer here.

#### 3-D) Est-il possible d'obtenir 1,5°C dans un environnement pessimiste? Évaluer le SSP5 dans le cadre de la politique d'atténuation$\mu=1$ d'ici 2060 pour imiter SSP5-1.9

In [ ]:
# SSP5-like / strong-mitigation stress test:
# start from init_states(p_pes), reuse the baseline saving path,
# assign ramp2060, then call update_path with p_pes.
# Compare with sim_base and report atmospheric warming in 2100.


> ✍️ You written answer here.